In [ ]:
!pip install nevergrad
from IPython.core.display import display
import nevergrad
import numpy
import matplotlib.pyplot as plt
import pandas
import seaborn
import sklearn
import sklearn.base
import sklearn.compose
import sklearn.inspection
import sklearn.model_selection
import sklearn.linear_model
import sklearn.pipeline
import sklearn.preprocessing
import scipy.stats
import typing

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-ames.git

# Régression linéaire

## Régression linéaire univariée

### Introduction

La régression linéaire est très intéressante à coder soi-même : elle n'est pas trop complexe mais permet de toucher à beaucoup de concepts du machine learning.

Nous allons continuer notre travail sur les données de prix de maisons.

Passons en python quelques secondes pour charger le dataset :

In [ ]:
train_df = pandas.read_csv("dataset-ames/train.csv", index_col="Id")

In [ ]:
display(train_df)

### Colonnes utilisées

Nous utiliserons dans cette démonstration seulement une caractéristique : `GrLivArea`, et `SalePrice` comme target.

In [ ]:
seaborn.jointplot(x="GrLivArea", y="SalePrice", data=train_df)
plt.show()

In [ ]:
### Standardize X
X_scaler = sklearn.preprocessing.StandardScaler()
X = X_scaler.fit_transform(train_df[["GrLivArea"]])

### Standardize y
Y_scaler = sklearn.preprocessing.StandardScaler()
Y = Y_scaler.fit_transform(train_df[["SalePrice"]])

### Exemple de Régression Linéaire
Pour comprendre l'évolution jointe du prix de vente et de la qualité globale d'une maison, on va se limiter à une hypothèse forte : on peut trouver une fonction affine qui modélise correctement la relation entre les entrées et les sorties. Pour rappel, une fonction affine est de la forme $h_\theta(x) = \theta_0 + \theta_1 x$.

Géométriquement, on pourra représenter cette fonction par une droite.

Par exemple, ici, on aimerait trouver l'hypothèse suivante :

In [ ]:
seaborn.regplot(x=X, y=Y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

Tout ça est prometteur, mais ici nous avons utilisé [`seaborn`](https://seaborn.pydata.org/) pour calculer les paramètres de l'hypothèse qui représente au mieux les données. Nous allons maintenant voir comment le faire nous même.

### Estimation des paramètres

Pour estimer $\theta_0$ et $\theta_1$, nous avons besoin de 2 éléments :

- une fonction de coût, qui nous dira pour des paramètres donnés $\theta_0$ et $\theta_1$ si l'on se débrouille bien ou non
- une méthode d'estimation des meilleurs paramètres étant donnée cette fonction de coût

### Prédictions

En prérequis au calcul de la fonction de coût, nous devons savoir calculer des prédictions.

Pour appliquer la transformation linéaire définie par $\theta_0$ et $\theta_1$ à $\mathbf{x}$, on utilise simplement la multiplication et l'addition de scalaires à un vecteur : $\mathbf{x}\theta_1 + \theta_0$

In [ ]:
def predict(X: numpy.ndarray, theta_0: float, theta_1: float) -> numpy.ndarray:
  return theta_0 + X * theta_1


X_example = numpy.array([[1], [2], [3]])
Y_example = numpy.array([[4], [10], [3]])
theta_0_example = 2
theta_1_example = 3

print(predict(X_example, theta_0_example, theta_1_example))

### Fonction de coût

La fonction de coût la plus utilisée est la moyenne des différences entre les points réels et ceux obtenus par l'hypothèse avec les paramètres courants, le tout au carré.

En notant $L$ la fonction de coût, $n$ le nombre d'exemples d'apprentissage, on peut écrire :

$$
  L(\theta_0, \theta_1) = \sum\frac{(\mathbf{x}\theta_1 + \theta_0 - \mathbf{y})^2}{2n}
$$

In [ ]:
def residuals(X: numpy.ndarray,
              Y: numpy.ndarray,
              theta_0: float,
              theta_1: float,
             ) -> numpy.ndarray:
  return predict(X, theta_0, theta_1) - Y

print("Residuals:",
      residuals(X_example, Y_example, theta_0_example, theta_1_example))

def cost(X: numpy.ndarray,
         Y: numpy.ndarray,
         theta_0: float,
         theta_1: float,
        ) -> numpy.float32:
  return (numpy.mean(residuals(X=X,
                               Y=Y,
                               theta_0=theta_0,
                               theta_1=theta_1)
                     ** 2)
          / 2)

print("Costs:",
      cost(X_example, Y_example, theta_0_example, theta_1_example))

### Optimisation des paramètres

Optimiser les paramètres revient à minimiser la fonction de coût :

$$\min_{\theta} \sum\frac{(\theta_0 + \mathbf{x}\theta_1 - \mathbf{y})^2}{2n}$$

Pour ce faire, il existe une formule directe que nous n'utiliserons pas dans ces travaux pratiques pour deux raisons : elle n'est pas applicable aux très grands datasets et la méthode que l'on va utiliser pourra être réutilisée pour la régression logistique et d'autres techniques (réseaux de neurones généraux, gradient boosted trees, etc).

Nous allons utiliser la descente de gradient. Cette méthode est itérative et fait des pas successifs en suivant la dérivée (pour maximiser) ou son opposé (pour minimiser) la fonction de perte. Dans le cas de la régression linéaire, elle converge vers l'optimum global (la meilleure solution).

En pseudo-code, l'algorithme est (avec les bons gradients calculés):

$$
\begin{aligned}
& \text{tant que ça n'a pas convergé :} \\
& \quad \theta_0 \leftarrow \theta_0 - \alpha \sum\frac{\theta_0 + \mathbf{x}\theta_1 - \mathbf{y}}{n} \\
& \quad \theta_1 \leftarrow \theta_1 - \alpha \frac{\mathbf{x}^T(\theta_0 + \mathbf{x}\theta_1 - \mathbf{y})}{n} \\
\end{aligned}
$$

où $\alpha$ est le pas d'apprentissage.

In [ ]:
def gradient(X: numpy.ndarray,
             Y: numpy.ndarray,
             theta_0: float,
             theta_1: float
            ) -> typing.Tuple[float, float]:
    r = residuals(X=X, Y=Y, theta_1=theta_1, theta_0=theta_0)
    g_0 = r.mean()
    g_1 = X.T.dot(r) / Y.size
    return g_0, g_1[0, 0]

### Descente de gradient

In [ ]:
def gradient_descent(X: numpy.ndarray,
                     Y: numpy.ndarray,
                     alpha: float = 0.1,
                     nb_iter: int = 1000,
                     epsilon: float = 10e-6
                    ) -> typing.Tuple[float, float, typing.List[float]]:
  theta_0 = numpy.random.rand(1)[0]
  theta_1 = numpy.random.rand(1)[0]
  print(f"Paramètres initiaux (aléatoires) : θ₀ = {theta_0:.2f}, θ₁ = {theta_1:.2f}")
  initial_cost = cost(X, Y, theta_0, theta_1)
  print(f"Coût initial : {initial_cost:.2f}")
  costs = [initial_cost]
  for i in range(nb_iter):
    g_0, g_1  = gradient(X, Y, theta_0, theta_1)
    theta_0 -= alpha * g_0
    theta_1 -= alpha * g_1
    costs.append(cost(X, Y, theta_0, theta_1))
    if costs[-2] - costs[-1] < epsilon:
      print(f"Arrêt de la procédure : pas de progrès suffisant à l'itération {i}")
      break
  print(f"Paramètres finaux : θ₀ = {theta_0:.2f}, θ₁ = {theta_1:.2f}")
  print(f"Coût final : {costs[-1]:.2f}")
  return theta_0, theta_1, costs

theta_0, theta_1, costs = gradient_descent(X, Y, 0.1, 1000)
print(f"Coûts successifs : {', '.join(map(str, costs))}")

Pour finir l'appel à la fonction gradient_descent

In [ ]:
theta_0, theta_1, costs = gradient_descent(X, Y, 0.01, 5000, 10e-6)

### Vérification de notre hypothèse

Vérifions maintenant le modèle appris avec quelques plots :

- valeurs prédites contre les valeurs réelles
- résiduels
- coûts d'entrainement

In [ ]:
inv_X = X_scaler.inverse_transform(X)
inv_Y = Y_scaler.inverse_transform(Y)
inv_Yh = Y_scaler.inverse_transform(predict(X, theta_0, theta_1))

Xs = numpy.arange(0, 6_000, 100).reshape(-1, 1)
Ys = Y_scaler.inverse_transform(predict(X_scaler.transform(Xs), theta_0, theta_1))
plt.plot(inv_X, inv_Y, 'b.', alpha=0.10)
plt.plot(Xs, Ys, 'r')
plt.title('Prédictions et valeurs réelles')
plt.xlabel('Surface habitable')
plt.ylabel('Prix de vente')
plt.show()

plt.plot(inv_X, inv_Y - inv_Yh, 'r.', alpha=0.10)
plt.title('Résiduels')
plt.xlabel('Surface habitable')
plt.ylabel('Résiduels')
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Coûts d'apprentissage pendant la descente de gradient")
plt.xlabel('Itérations')
plt.ylabel('Erreur')
plt.show()

### Correction de l'asymétrie positive

Il est intéressant de corriger l'asymétrie positive de la variable de sortie si on l'a remarquée : on peut utiliser pour cela la fonction [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) et sa transformation inverse [`numpy.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html).

In [ ]:
seaborn.distplot(train_df[["SalePrice"]], fit=scipy.stats.norm)
plt.title('Distribution de SalePrice avant normalisation')
plt.show()

Y_scaler_log = sklearn.preprocessing.StandardScaler()
Y = Y_scaler_log.fit_transform(numpy.log1p(train_df[["SalePrice"]]))

seaborn.distplot(Y, fit=scipy.stats.norm)
plt.title('Distribution de SalePrice après normalisation')
plt.show()

In [ ]:
theta_0, theta_1, costs = gradient_descent(X, Y, 0.01, 5000, 10e-6)

inv_X = X_scaler.inverse_transform(X)
inv_Y = numpy.expm1(Y_scaler_log.inverse_transform(Y))
inv_Yh = numpy.expm1(Y_scaler_log.inverse_transform(predict(X,
                                                            theta_0,
                                                            theta_1)))

Xs = numpy.arange(0, 6_000, 100).reshape(-1, 1)
Ys = numpy.expm1(Y_scaler_log.inverse_transform(predict(X_scaler.transform(Xs), theta_0, theta_1)))
plt.plot(inv_X, inv_Y, 'b.', alpha=0.10)
plt.plot(Xs, Ys, 'r')
plt.title('Prédictions et valeurs réelles')
plt.xlabel('Surface habitable')
plt.ylabel('Prix de vente')
plt.show()

plt.plot(inv_X, inv_Y - inv_Yh, 'r.', alpha=0.10)
plt.title('Résiduels')
plt.xlabel('Surface habitable')
plt.ylabel('Résiduels')
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Coûts d'apprentissage pendant la descente de gradient")
plt.xlabel('Itérations')
plt.ylabel('Erreur')
plt.show()

## Régression linéaire multivariée

Pour poursuivre cette démonstration, nous allons reprendre un prétraitement complet effectué sur la totalité des données AMES (et non pas seulement la colonne `GrLiveArea`).

In [ ]:
def preprocess(train_file: str, test_file: str) -> numpy.ndarray:
    train_X = pandas.read_csv(train_file, index_col="Id")
    test_X = pandas.read_csv(test_file, index_col="Id")

    train_Y = train_X[["SalePrice"]]
    train_X = train_X.drop(columns="SalePrice")

    all_X = pandas.concat([train_X, test_X])

    # Fill with median
    cols_1 = ["LotFrontage"]
    all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

    # Fill with mode
    cols_2 = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st",
             "Exterior2nd", "SaleType", "Utilities"]
    all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

    # Fill with 0
    cols_4 = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1",
              "BsmtFinSF2", "BsmtFullBath", "BsmtHalfBath", "BsmtUnfSF",
              "MasVnrArea", "TotalBsmtSF"]
    all_X[cols_4] = all_X[cols_4].fillna(0)

    # Other fills
    cols_5 = ["Functional"]
    all_X[cols_5] = all_X[cols_5].fillna("Typ")

    # On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
    all_X = all_X.fillna("NA")

    # On transforme le codage numérique en string afin que ce soit traité comme
    # une variable catégorielle
    cols_numerical2label = ['MSSubClass']
    all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

    quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
    quality_columns = ["BsmtCond", "BsmtQual", "ExterCond", "ExterQual",
                       "FireplaceQu", "GarageCond", "GarageQual", "HeatingQC",
                       "KitchenQual", "PoolQC"]
    street_mapping = dict(NA=0, Grvl=1, Pave=2)
    bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

    replace_mapping = dict(
      Alley=street_mapping,
      BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
      BsmtFinType1=bsmt_fin_mapping,
      BsmtFinType2=bsmt_fin_mapping,
      Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
      LandSlope=dict(Sev=1, Mod=2, Gtl=3),
      LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
      PavedDrive=dict(NA=0, N=1, P=2, Y=3),
      Street=dict(Grvl=1, Pave=2),
      Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
    )

    for quality_column in quality_columns:
      replace_mapping[quality_column] = quality_mapping

    all_X.replace(replace_mapping, inplace=True)

    print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

    dummies = pandas.get_dummies(all_X)
    return (dummies.iloc[:train_X.shape[0], :].values,
            train_Y.values,
            dummies.iloc[train_X.shape[0]:, :].values,
            dummies.columns)

In [ ]:
X_train, Y_train, X_test, columns = preprocess("dataset-ames/train.csv",
                                               "dataset-ames/test.csv")

### Normalisation de la variable de sortie

Utilisons [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) pour centrer et réduire les cibles dans la nouvelle variable `Y_train_scaled`.

In [ ]:
Y_scaler = sklearn.preprocessing.StandardScaler()
Y_train_scaled = Y_scaler.fit_transform(Y_train)

### Entraînement d'un modèle de régression linéaire simple

In [ ]:
linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(X_train, Y_train_scaled)
print(linear_regression.coef_)

### Validation croisée

In [ ]:
def score(model: sklearn.base.BaseEstimator,
          X: numpy.ndarray = X_train,
          Y: numpy.ndarray = Y_train_scaled
          ):
  scores = sklearn.model_selection.cross_val_score(
    model,
    X,
    Y,
    cv=5,
    scoring="r2")
  return sum(scores) / len(scores)


def score(model: sklearn.base.BaseEstimator,
          X: numpy.ndarray = X_train,
          Y: numpy.ndarray = Y_train_scaled
          ) -> float:
  pipeline = sklearn.compose.TransformedTargetRegressor(
      regressor=model,
      transformer=sklearn.preprocessing.StandardScaler())
  scores = sklearn.model_selection.cross_val_score(
    pipeline,
    X,
    Y,
    cv=5,
    scoring="r2")
  return sum(scores) / len(scores)


score(sklearn.linear_model.LinearRegression())

### Régularisation L1 et L2

In [ ]:
score_l1 = score(sklearn.linear_model.Lasso())
score_l2 = score(sklearn.linear_model.Ridge())
score_l1_l2 = score(sklearn.linear_model.ElasticNet())
print(f"Scores l1, l2 et l1+l2 :", score_l1, score_l2, score_l1_l2)

### Recherche d'hyper-paramètres

`scikit-learn` contient quelques algorithmes de recherche d'hyperparamètres, mais ceux-ci sont assez limités (random search & grid search). Plusieurs librairies tierces sont disponibles pour fournir cette fonctionnalité.

In [ ]:
# Modèle simple
model = sklearn.linear_model.Ridge()
params = dict(
    alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3]
)
rscv = sklearn.model_selection.RandomizedSearchCV(model, params, scoring="r2",
                                                  n_iter=6, verbose=0)
search = rscv.fit(X_train, Y_train_scaled)
print("="*80)
print(f"Paramètres trouvés par RSCV : {search.best_params_}")
print(f"score: {search.best_score_}")
print("="*80)

# Modèle complexe qui utilise une transformation des cibles
model = sklearn.compose.TransformedTargetRegressor(
    regressor=sklearn.linear_model.Ridge(),
    transformer=sklearn.preprocessing.StandardScaler())
params = dict(
    regressor__alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3]
)
rscv = sklearn.model_selection.RandomizedSearchCV(model, params, scoring="r2",
                                                  n_iter=6, verbose=0)
search = rscv.fit(X_train, Y_train)
print("=" * 80)
print(f"Paramètres trouvés par RSCV : {search.best_params_}")
print(f"score: {search.best_score_}")
print("="  *80)

# Avec Nevergrad
def loss(alpha: float) -> float:
  return -score(sklearn.linear_model.Ridge(alpha=alpha))


parametrization = nevergrad.p.Instrumentation(
    alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization,
                                       budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)
best_model.fit(X_train, Y_train_scaled)
print()
print("=" * 80)
print(f"Paramètres trouvés par Nevergrad : {recommendation.kwargs}")
print(f"score: {score(best_model)}")
print("=" * 80)

### Importance des caractéristiques

Utilisons la mesure d'importance par permutation de scikit-learn ([`sklearn.inspection.permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)) pour évaluer quelles caractéristiques sont les plus utiles au meilleur modèle entraîné après la recherche d'hyper-paramètres.

In [ ]:
feature_importances = sklearn.inspection.permutation_importance(
    best_model, X_train, Y_train_scaled, n_repeats=50)

In [ ]:
series = pandas.Series(feature_importances.importances_mean, index=columns)
series = series.sort_values(ascending=False).iloc[:10]
series.plot.bar()

### Caractéristiques polynômiales

Outre le travail d'extension manuel des caractéristiques, il est possible d'utiliser scikit-learn pour étendre la matrice d'apprentissage avec des caractéristiques polynômiales et sa classe [`sklearn.preprocessing.PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

In [ ]:
top10 = (-feature_importances.importances_mean).argsort()[:10]


def poly_features(array: numpy.ndarray) -> numpy.ndarray:
  polynomial_features = sklearn.preprocessing.PolynomialFeatures(
      2, interaction_only=True)
  return polynomial_features.fit_transform(array[:, top10])


X_train_poly = poly_features(X_train)
X_test_poly = poly_features(X_test)



parametrization = nevergrad.p.Instrumentation(
    alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization,
                                       budget=50)
recommendation = optimizer.minimize(loss,verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)
best_model.fit(X_train_poly, Y_train_scaled)

print()
print("=" * 80)
print(f"Paramètres trouvés par Nevergrad : {recommendation.kwargs}")
print(f"score: {score(best_model)}")
print("=" * 80)